In [1]:
!unzip /content/archive.zip



Archive:  /content/archive.zip
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_10.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_100.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_101.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_102.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_103.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_104.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_105.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_106.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_107.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_108.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_109.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lumpy_Skin_11.png  
  inflating: Lumpy Skin Images Dataset/Lumpy Skin/Lu

In [2]:
from google.colab import files
import os

os.makedirs("/content/Lumpy Skin Images Dataset/Lumpy Skin", exist_ok=True)
os.makedirs("/content/Lumpy Skin Images Dataset/Normal Skin", exist_ok=True)

print("Upload lumpy images:")
uploaded = files.upload()  # Upload lumpy images
for name in uploaded.keys():
    os.rename(name, f"/content/Lumpy Skin Images Dataset/Lumpy Skin/{name}")

print("Upload normal images:")
uploaded = files.upload()  # Upload normal images
for name in uploaded.keys():
    os.rename(name, f"/content/Lumpy Skin Images Dataset/Normal Skin/{name}")


Upload lumpy images:


KeyboardInterrupt: 

In [3]:
!pip install tensorflow opencv-python


In [2]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# Load images and labels
def load_images(folder, label, size=(224, 224)):
    images = []
    labels = []
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        img = cv2.imread(path)
        if img is not None:
            img = cv2.resize(img, size)
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# Load dataset
lumpy_imgs, lumpy_labels = load_images("Lumpy Skin Images Dataset/Lumpy Skin", 1)
normal_imgs, normal_labels = load_images("Lumpy Skin Images Dataset/Normal Skin", 0)

X = np.concatenate((lumpy_imgs, normal_imgs), axis=0)
y = np.concatenate((lumpy_labels, normal_labels), axis=0)

# Preprocess
X = preprocess_input(X)
y = to_categorical(y, 2)

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Load VGG16
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

# Build model
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dense(2, activation='softmax')
])

# Compile and train
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=8)

# Save the model
model.save("model.keras")




Epoch 1/10


103/103 [==============================] - 229s 2s/step - loss: 3.3732 - accuracy: 0.8168 - val_loss: 2.4123 - val_accuracy: 0.8439
Epoch 2/10
103/103 [==============================] - 186s 2s/step - loss: 0.5870 - accuracy: 0.9548 - val_loss: 2.1942 - val_accuracy: 0.8732
Epoch 3/10
103/103 [==============================] - 515s 5s/step - loss: 0.1836 - accuracy: 0.9915 - val_loss: 2.1625 - val_accuracy: 0.8829
Epoch 4/10
103/103 [==============================] - 187s 2s/step - loss: 0.1634 - accuracy: 0.9927 - val_loss: 1.9552 - val_accuracy: 0.8829
Epoch 5/10
103/103 [==============================] - 273s 3s/step - loss: 0.2618 - accuracy: 0.9927 - val_loss: 1.9879 - val_accuracy: 0.8829
Epoch 6/10
103/103 [==============================] - 189s 2s/step - loss: 0.1234 - accuracy: 0.9963 - val_loss: 1.7662 - val_accuracy: 0.8829
Epoch 7/10
103/103 [==============================] - 167s 2s/step - loss: 0.0906 - accuracy: 0.9939 - val_loss: 1.7642 - val_accuracy: 0.

In [5]:
from google.colab import files
files.download("model.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
uploaded = files.upload()  # Upload one image to test
image_path = list(uploaded.keys())[0]


In [7]:
from tensorflow.keras.models import load_model

# Load model
model = load_model("model.h5")

# Predict
def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (224, 224))
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    pred = model.predict(img)
    class_names = ['Normal', 'Lumpy']
    return class_names[np.argmax(pred)]

result = predict_image(image_path)
print("Prediction:", result)


NameError: name 'image_path' is not defined

In [6]:
import matplotlib.pyplot as plt

history = model.fit(...)  # store this during training

# Accuracy plot
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Loss plot
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


ValueError: Unrecognized data type: x=Ellipsis (of type <class 'ellipsis'>)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = np.argmax(y_val, axis=1)
y_pred = np.argmax(model.predict(X_val), axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Lumpy"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc

y_scores = model.predict(X_val)[:, 1]  # score for class 'Lumpy'
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()


In [ ]:
import seaborn as sns

labels = np.concatenate((lumpy_labels, normal_labels))
sns.countplot(x=labels)
plt.xticks([0, 1], ['Normal', 'Lumpy'])
plt.title("Class Distribution")
plt.show()
